# Day 9 FAISS GPU build — Google Colab T4

This notebook builds the locked BGE-M3 `IndexFlatIP` artifact on a Colab T4.

Before running, copy the `day9-dense` repository snapshot to:
`MyDrive/financial-assistant-day9/repo/`

The notebook keeps the release lock, corpus, model cache, logs, observation, and index under Google Drive. It does not delete or quarantine data.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request

from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/financial-assistant-day9')
REPO_ROOT = DRIVE_ROOT / 'repo'
FINGERPRINT = '422df141c935d46bfd14302abec50f32380e6e4c012159f8ad0ae5560c8a446a'
LOCK_PATH = REPO_ROOT / 'data/qa/week1_pilot_422df141c935/dataset-pilot-v1.json'
CORPUS_DIR = REPO_ROOT / f'data/indexes/dense-day9-a/{FINGERPRINT}/corpus'
OUTPUT_ROOT = REPO_ROOT / f'data/indexes/dense-day9-a/{FINGERPRINT}/encoders'
OBSERVATION_PATH = REPO_ROOT / 'artifacts/evaluations/day9/bge-m3-faiss-gpu-build.json'
LOG_PATH = REPO_ROOT / 'artifacts/evaluations/day9/bge-m3-faiss-gpu-build.log'
QUARANTINE_ROOT = REPO_ROOT / 'data/quarantine/day9-cleanup'
HF_HOME = DRIVE_ROOT / 'hf-cache'
MODEL_REVISION = '5617a9f61b028005a4858fdac845db406aefb181'
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME / 'hub')

def run_checked(cmd, **kwargs):
    """subprocess.run with check=True that prints captured stdout/stderr before raising."""
    kwargs.setdefault('capture_output', True)
    kwargs.setdefault('text', True)
    result = subprocess.run(cmd, **kwargs)
    if result.returncode != 0:
        print(f'--- command failed: {cmd} ---')
        if result.stdout:
            print('stdout:\n' + result.stdout)
        if result.stderr:
            print('stderr:\n' + result.stderr)
        result.check_returncode()
    return result

_required = [
    REPO_ROOT / 'pyproject.toml',
    REPO_ROOT / 'README.md',
    REPO_ROOT / 'uv.lock',
    LOCK_PATH,
    CORPUS_DIR / 'manifest.json',
]
if not LOCK_PATH.exists():
    raise FileNotFoundError(f'Missing required Drive artifact: {LOCK_PATH}')
_lock = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
_required.append(REPO_ROOT / _lock['release_path'] / 'manifest.json')
for _parquet in ('documents.parquet', 'tables.parquet', 'cells.parquet'):
    _required.append(REPO_ROOT / _lock['release_path'] / _parquet)
_required.append(REPO_ROOT / _lock['gate_result_path'])
for required in _required:
    if not required.exists():
        raise FileNotFoundError(f'Missing required Drive artifact: {required}')
if _lock['dataset_fingerprint'] != FINGERPRINT:
    raise RuntimeError('Release lock fingerprint does not match the notebook fingerprint')
print(f'Repository: {REPO_ROOT}')
print(f'Corpus: {CORPUS_DIR}')
print(f'Output: {OUTPUT_ROOT}')

## Install an isolated GPU environment

The project requires Python 3.11–<3.12. FAISS GPU 1.14.2 is used because the 1.15.0 build currently resolves only for Python 3.12.

In [ ]:
MAMBA_ROOT = Path('/content/micromamba')
MAMBA_BIN = MAMBA_ROOT / 'bin/micromamba'
ENV_PREFIX = MAMBA_ROOT / 'envs/financial-faiss-gpu'
ENV_PYTHON = ENV_PREFIX / 'bin/python'
MAMBA_BIN.parent.mkdir(parents=True, exist_ok=True)
if not MAMBA_BIN.exists():
    archive = Path('/content/micromamba.tar.bz2')
    urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest', archive)
    run_checked(['tar', '-xvjf', str(archive), '-C', str(MAMBA_BIN.parent), '--strip-components=1', 'bin/micromamba'])
if not ENV_PYTHON.exists():
    run_checked([
        str(MAMBA_BIN), 'create', '-y', '-p', str(ENV_PREFIX),
        '-c', 'pytorch', '-c', 'nvidia', '-c', 'conda-forge',
        'python=3.11', 'faiss-gpu=1.14.2',
    ])
run_checked([str(MAMBA_BIN), 'run', '-p', str(ENV_PREFIX), 'python', '-m', 'pip', 'install', '--upgrade', 'pip', 'uv'])
exported = run_checked([
    str(MAMBA_BIN), 'run', '-p', str(ENV_PREFIX), 'uv', 'export',
    '--frozen', '--no-dev', '--no-emit-project', '--no-hashes',
], cwd=REPO_ROOT).stdout
requirements = '\n'.join(line for line in exported.splitlines() if not line.lower().startswith('faiss-cpu')) + '\n'
requirements_path = Path('/content/financial-assistant-gpu-requirements.txt')
requirements_path.write_text(requirements, encoding='utf-8')
run_checked([str(MAMBA_BIN), 'run', '-p', str(ENV_PREFIX), 'python', '-m', 'pip', 'install', '-r', str(requirements_path)])
run_checked([str(MAMBA_BIN), 'run', '-p', str(ENV_PREFIX), 'python', '-m', 'pip', 'install', '--no-deps', '-e', str(REPO_ROOT)])
print(f'Environment ready: {ENV_PREFIX}')

## Cache the pinned BGE-M3 snapshot on Drive

The actual build uses local-only model loading. This cell performs the one-time pinned download only when the Drive cache is absent.

In [ ]:
model_marker = HF_HOME / 'hub/models--BAAI--bge-m3'
if not model_marker.exists():
    download_code = (
        'from huggingface_hub import snapshot_download; '
        "snapshot_download(repo_id='BAAI/bge-m3', revision='" + MODEL_REVISION + "', cache_dir='" + str(HF_HOME / 'hub') + "')"
    )
    subprocess.run([str(ENV_PYTHON), '-c', download_code], check=True, env=os.environ.copy())
else:
    print(f'Using cached BGE-M3 snapshot: {model_marker}')
print(f'HF cache: {HF_HOME}')

## T4 / FAISS GPU preflight

In [ ]:
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], check=True, capture_output=True, text=True).stdout.strip()
gpu_count = int(subprocess.run([str(ENV_PYTHON), '-c', 'import faiss; print(faiss.get_num_gpus())'], check=True, capture_output=True, text=True, env=os.environ.copy()).stdout.strip())
print(f'GPU: {gpu_name}')
print(f'FAISS GPUs: {gpu_count}')
if 'T4' not in gpu_name:
    raise RuntimeError(f'Expected a Colab T4 runtime, got: {gpu_name}')
if gpu_count <= 0:
    raise RuntimeError('FAISS reports no CUDA GPU; stopping before build')

## Cleanup safety check (dry-run only)

In [ ]:
cleanup_cmd = [
    str(ENV_PYTHON), '-m', 'financial_report_qa.cli', 'retrieval',
    'cleanup-day9-data', '--repo-root', str(REPO_ROOT),
    '--quarantine-root', str(QUARANTINE_ROOT),
]
cleanup_result = subprocess.run(cleanup_cmd, cwd=REPO_ROOT, capture_output=True, text=True, env=os.environ.copy())
print(cleanup_result.stdout)
if cleanup_result.returncode != 0:
    raise RuntimeError(cleanup_result.stderr or 'Cleanup dry-run failed')
cleanup_entries = [json.loads(line) for line in cleanup_result.stdout.splitlines() if line.strip()]
blocked = [entry for entry in cleanup_entries if entry.get('status') == 'blocked']
if blocked:
    raise RuntimeError(f'Cleanup has blocked candidates; inspect output before building: {blocked}')
print('Cleanup dry-run passed; no mutation requested.')

## Build BGE-M3 FAISS GPU index with streamed log

In [ ]:
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
build_cmd = [
    str(ENV_PYTHON), '-u', '-m', 'financial_report_qa.cli', 'retrieval',
    'build-dense-index', '--release-lock', str(LOCK_PATH),
    '--corpus-dir', str(CORPUS_DIR), '--encoder', 'bge-m3',
    '--output-root', str(OUTPUT_ROOT), '--observation-path', str(OBSERVATION_PATH),
    '--local-files-only', '--faiss-device', 'cuda',
]
build_env = os.environ.copy()
build_env['PYTHONPATH'] = str(REPO_ROOT / 'src')
process = subprocess.Popen(build_cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=build_env)
with LOG_PATH.open('w', encoding='utf-8') as log_file:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Dense GPU build failed with exit code {return_code}; see {LOG_PATH}')
print(f'Build log: {LOG_PATH}')

## Show the last recorded progress line

In [ ]:
progress_pattern = re.compile(r'dense-build: (\d+)/(\d+) vectors, ([0-9.]+)s, ([0-9.]+) vectors/s')
progress_lines = [line.strip() for line in LOG_PATH.read_text(encoding='utf-8').splitlines() if progress_pattern.search(line)]
if progress_lines:
    print(progress_lines[-1])
else:
    print('No batch progress line found; inspect the full log:', LOG_PATH)

## Verify locked artifacts

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

release_lock = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
release_lock_sha256 = sha256_file(LOCK_PATH)
corpus_manifest = json.loads((CORPUS_DIR / 'manifest.json').read_text(encoding='utf-8'))
observation = json.loads(OBSERVATION_PATH.read_text(encoding='utf-8'))
if release_lock.get('dataset_fingerprint') != FINGERPRINT:
    raise RuntimeError('Release lock fingerprint mismatch')
if observation.get('dataset_fingerprint') != FINGERPRINT or observation.get('faiss_device') != 'cuda' or observation.get('faiss_gpu_count', 0) <= 0:
    raise RuntimeError('Observation is not the requested CUDA build')
if corpus_manifest.get('dataset_fingerprint') != FINGERPRINT or corpus_manifest.get('release_lock_sha256') != release_lock_sha256:
    raise RuntimeError('Corpus manifest identity mismatch')
index_dirs = sorted(path for path in OUTPUT_ROOT.iterdir() if path.is_dir())
if not index_dirs:
    raise RuntimeError(f'No persisted index directory under {OUTPUT_ROOT}')
for index_dir in index_dirs:
    index_path = index_dir / 'index.faiss'
    manifest_path = index_dir / 'manifest.json'
    if not index_path.is_file() or not manifest_path.is_file():
        raise RuntimeError(f'Incomplete index artifact: {index_dir}')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('dataset_fingerprint') != FINGERPRINT:
        raise RuntimeError(f'Index fingerprint mismatch: {index_dir}')
    if manifest.get('release_lock_sha256') != release_lock_sha256:
        raise RuntimeError(f'Index release-lock mismatch: {index_dir}')
    if manifest.get('index_type') != 'IndexFlatIP' or manifest.get('encoder', {}).get('name') != 'bge-m3' or manifest.get('encoder', {}).get('revision') != MODEL_REVISION:
        raise RuntimeError(f'Index type is not IndexFlatIP: {index_dir}')
    if manifest.get('document_count') != corpus_manifest.get('document_count'):
        raise RuntimeError(f'Document count mismatch: {index_dir}')
    if manifest.get('artifact_sha256', {}).get('index.faiss') != sha256_file(index_path):
        raise RuntimeError(f'index.faiss hash mismatch: {index_dir}')
print('PASS: CUDA observation, lock/fingerprint, IndexFlatIP, counts, and index hashes verified.')
print(f'Observation: {OBSERVATION_PATH}')
print(f'Log: {LOG_PATH}')
print(f'Index root: {OUTPUT_ROOT}')